In [17]:
from pulao.logging import init_logging
import logging
from time import sleep
from typing import Tuple

import pandas as pd

from IPython.display import display

from pulao.constant import  FractalType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#

# region 2. Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]

# 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[(df_cbar["index"]>=row.start_index) & (df_cbar["index"]<=row.end_index)]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs = []
ys = []
texts = []
for i, row in enumerate(df_swing.itertuples()):
    xs += [row.start_datetime, row.end_datetime, None]
    if row.direction == SwingDirection.UP:
        start_price = df_cbar.at[row.start_index,"low_price"]
        end_price = df_cbar.at[row.end_index,"high_price"]
    else:
        start_price = df_cbar.at[row.start_index,"high_price"]
        end_price = df_cbar.at[row.end_index,"low_price"]
    ys += [start_price, end_price, None]
    texts += [row.start_index, row.end_index, None]


fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    name='swing segments',
    mode='lines+text',    # lines+text 支持显示文字
    line=dict(width=2, color='orange'),
    text=texts,
    textposition='top center',  # 文字位置
))

# title = 'CBar Chart - K线合并处理'
title = 'CBar Chart - 分形重叠处理'
# title = 'CBar Chart - 次级别处理'
# title = 'CBar Chart - 连续同级别同向高低点处理'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)
# fig2 = None
fig.show()


# endregion

{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 07:01:29"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 07:01:29"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 07:01:29"}
{"fractal": "Fractal(left=CBar(index=0, start_index=0, end_index=1, high_price=942.2349853515625, low_price=935.9240112304688, fractal_type=0), middle=CBar(index=1, start_index=2, end_index=2, high_price=939.052001953125, low_price=930.2979736328125, fractal_type=-1), right=CBar(index=2, start_index=3, end_index=3, high_price=951.2730102539062, low_price=937.4929809570312, fractal_type=0))", "df_swing_height": 1, "event": "情况2，最后3根bar能组成分形。第一次构建，直接添加。", "level": "debug", "logger": "pulao.swing.swing_manager", "time": "2025-11-25 07:01:29"}
{"active_swing": "Swing(index=0, direction=1, 

In [ ]:
from pulao.logging import init_logging
import logging
from IPython.display import display

from pulao.constant import FractalType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value,datetime=datetime,turnover=money,open_price=open,close_price=close,high_price=high,low_price=low,volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
# trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#
display(swing_manager.df_swing)

cbar_manager.df_cbar

# swing_manager.df_swing

In [14]:
from pulao.constant import SwingDirection

# display(df_cbar)
display(df_swing)

# df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
# df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
#
# for i, row in enumerate(df_swing.itertuples()):
#     df = df_cbar[(df_cbar["index"]>=row.start_index) & (df_cbar["index"]<=row.end_index)]
#     start_datetime = df.iloc[0]["datetime"]
#     end_datetime = df.iloc[-1]["datetime"]
#     df_swing.at[i, "start_datetime"] = start_datetime
#     df_swing.at[i, "end_datetime"] = end_datetime
#
# xs=[]
# ys=[]
# for i, row in enumerate(df_swing.itertuples()):
#     xs += [row.start_datetime, row.end_datetime, None]
#     if row.direction == SwingDirection.UP:
#         start_price = df_cbar.at[row.start_index,"low_price"]
#         end_price = df_cbar.at[row.end_index,"high_price"]
#     else:
#         start_price = df_cbar.at[row.start_index,"high_price"]
#         end_price = df_cbar.at[row.end_index,"low_price"]
#     ys += [start_price, end_price, None]
# display(xs,ys)


,index,start_index,end_index,high_price,low_price,direction,is_completed,start_datetime,end_datetime
0,0,1,7,961.794983,930.297974,1,True,2025-01-01 01:00:00,2025-01-01 07:00:00
1,1,7,13,980.258972,947.198975,-1,True,2025-01-01 07:00:00,2025-01-01 13:00:00
2,2,13,32,988.528015,948.010010,1,True,2025-01-01 13:00:00,2025-01-02 08:00:00
3,3,32,42,965.278015,921.375000,-1,True,2025-01-02 08:00:00,2025-01-02 18:00:00
4,4,42,72,955.924988,906.143005,1,False,2025-01-02 18:00:00,2025-01-04 00:00:00
